# Data Understanding 3: `installments_payments`

Every scheduled instalment on customers' **previous Home Credit loans**, with what was actually paid and when. It is the most direct record of repayment behaviour in the dataset.

Each table is profiled with the same five questions:

1. **Grain:** what does one row represent?
2. **Real rows:** what does the data actually look like?
3. **Types:** what type is each column, and what does it really mean?
4. **Missing data:** where are the blanks, and why?
5. **Ranges:** do the values make sense, and what looks wrong?

**Performance note:** this table has 13.6 million rows. Every query aggregates inside DuckDB; nothing loads the full table into pandas.

## Setup

DuckDB queries the CSV files directly, so no table is loaded fully into memory. Only small, aggregated results are passed to pandas for display. Run the cells top to bottom.

In [1]:
import duckdb
import pandas as pd

INST = 'data/raw/installments_payments.csv'

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)

---
## Q1. Grain

There is no single row ID, so the grain is tested with combinations of columns (a **composite key**).

In [2]:
duckdb.sql(f'''
    SELECT
        COUNT(*)                   AS total_rows,
        COUNT(DISTINCT SK_ID_PREV) AS unique_prev_loans,
        COUNT(DISTINCT SK_ID_CURR) AS unique_customers
    FROM '{INST}'
''').show()

┌────────────┬───────────────────┬──────────────────┐
│ total_rows │ unique_prev_loans │ unique_customers │
│   int64    │       int64       │      int64       │
├────────────┼───────────────────┼──────────────────┤
│   13605401 │            997752 │           339587 │
└────────────┴───────────────────┴──────────────────┘



**Theory:** one row = one instalment of one loan. Then loan + instalment number should be unique.

In [3]:
duckdb.sql(f'''
    WITH combos AS (
        SELECT DISTINCT SK_ID_PREV, NUM_INSTALMENT_NUMBER
        FROM '{INST}'
    )
    SELECT COUNT(*) AS unique_loan_instalment FROM combos
''').show()

┌────────────────────────┐
│ unique_loan_instalment │
│         int64          │
├────────────────────────┤
│               12861994 │
└────────────────────────┘



Adding the schedule version to the key:

In [4]:
duckdb.sql(f'''
    WITH combos AS (
        SELECT DISTINCT SK_ID_PREV, NUM_INSTALMENT_NUMBER, NUM_INSTALMENT_VERSION
        FROM '{INST}'
    )
    SELECT COUNT(*) AS unique_loan_instalment_version FROM combos
''').show()

┌────────────────────────────────┐
│ unique_loan_instalment_version │
│             int64              │
├────────────────────────────────┤
│                       12951918 │
└────────────────────────────────┘



**Findings**

| Key | Unique combinations | Gap to 13,605,401 rows |
|---|---|---|
| loan + instalment | 12,861,994 | 743,407 |
| loan + instalment + version | 12,951,918 | 653,483 |

- 339,587 customers, 997,752 previous loans, 13,605,401 rows: about 13.6 rows per loan.
- About 90,000 repeats come from **schedule changes** (a new `NUM_INSTALMENT_VERSION`).
- The remaining ~650,000 are **split payments**: one instalment paid in several parts.
- **Grain: one row = one payment toward one instalment (of one version) of one previous loan.**
- Treating each row as a full instalment would turn split payments into fake underpayments. Payments must be aggregated **per instalment first**, then per customer.

---
## Q2. Real rows

One loan read in order, like a bank statement.

In [5]:
duckdb.sql(f'''
    SELECT
        *,
        DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT AS days_late
    FROM '{INST}'
    WHERE SK_ID_PREV = 2050653
    ORDER BY NUM_INSTALMENT_NUMBER, DAYS_ENTRY_PAYMENT
''').df()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,days_late
0,2050653,109083,1.0,1,-388.0,-396.0,51290.145,51290.145,-8.0
1,2050653,109083,1.0,2,-358.0,-361.0,51290.145,51290.145,-3.0
2,2050653,109083,1.0,3,-328.0,-361.0,51290.145,3169.710,-33.0
3,2050653,109083,1.0,3,-328.0,-323.0,51290.145,48120.435,5.0
4,2050653,109083,2.0,4,-298.0,-309.0,128709.855,180000.000,-11.0
5,2050653,109083,1.0,4,-298.0,-309.0,51290.145,180000.000,-11.0
6,2050653,109083,3.0,5,-268.0,-267.0,43517.340,43517.340,1.0
7,2050653,109083,3.0,6,-238.0,-240.0,43517.340,43517.340,-2.0
8,2050653,109083,4.0,7,-208.0,-218.0,729694.170,729694.170,-10.0


**Reading the dates:** more negative means further in the past, so comparing negatives by eye is error-prone. `days_late = paid - due`: **positive = late, negative = early**.

**Findings: loan 2050653**

- A good payer: mostly early, only twice slightly late (5 days and 1 day).
- **Instalment 3** was split: 3,169.71 + 48,120.435 = 51,290.145, exactly the amount due.
- **Instalment 4** has two versions (due 51,290.145 and 128,709.855). One payment of **180,000** covers both, but is **recorded on both rows**. Summing `AMT_PAYMENT` per instalment would count 360,000.
- **Instalment 7** pays 729,694 at once, then nothing: an early payoff.
- Payment features must avoid double-counting the same payment across versions.

---
## Q3. Column types and meaning

In [6]:
duckdb.sql(f"DESCRIBE SELECT * FROM '{INST}'").df()

,column_name,column_type,null,key,default,extra
0,SK_ID_PREV,BIGINT,YES,None,None,None
1,SK_ID_CURR,BIGINT,YES,None,None,None
2,NUM_INSTALMENT_VERSION,DOUBLE,YES,None,None,None
3,NUM_INSTALMENT_NUMBER,BIGINT,YES,None,None,None
4,DAYS_INSTALMENT,DOUBLE,YES,None,None,None
5,DAYS_ENTRY_PAYMENT,DOUBLE,YES,None,None,None
6,AMT_INSTALMENT,DOUBLE,YES,None,None,None
7,AMT_PAYMENT,DOUBLE,YES,None,None,None


**Findings**

| Role | Columns |
|---|---|
| IDs | `SK_ID_PREV`, `SK_ID_CURR` |
| Dates (days before application) | `DAYS_INSTALMENT` (due), `DAYS_ENTRY_PAYMENT` (paid) |
| Money | `AMT_INSTALMENT` (due), `AMT_PAYMENT` (paid) |
| Counters / labels | `NUM_INSTALMENT_NUMBER`, `NUM_INSTALMENT_VERSION` |

- No text columns.
- `NUM_INSTALMENT_VERSION` is stored as a decimal, but it is a **label** ("second schedule"), not a quantity. Averaging it means nothing.
- The columns come in **due vs paid pairs**, which is what the repayment features will compare.

---
## Q4. Missing data

In [7]:
inst_summary = duckdb.sql(f"SUMMARIZE SELECT * FROM '{INST}'").df()
inst_summary[['column_name', 'column_type', 'null_percentage']].sort_values('null_percentage', ascending=False)

,column_name,column_type,null_percentage
7,AMT_PAYMENT,DOUBLE,0.02
5,DAYS_ENTRY_PAYMENT,DOUBLE,0.02
0,SK_ID_PREV,BIGINT,0.00
1,SK_ID_CURR,BIGINT,0.00
3,NUM_INSTALMENT_NUMBER,BIGINT,0.00
2,NUM_INSTALMENT_VERSION,DOUBLE,0.00
4,DAYS_INSTALMENT,DOUBLE,0.00
6,AMT_INSTALMENT,DOUBLE,0.00


Do the two payment columns go missing together?

In [8]:
duckdb.sql(f'''
    SELECT
        SUM(CASE WHEN DAYS_ENTRY_PAYMENT IS NULL THEN 1 ELSE 0 END)                     AS no_payment_date,
        SUM(CASE WHEN AMT_PAYMENT IS NULL THEN 1 ELSE 0 END)                            AS no_payment_amount,
        SUM(CASE WHEN DAYS_ENTRY_PAYMENT IS NULL AND AMT_PAYMENT IS NULL THEN 1 ELSE 0 END) AS both_missing
    FROM '{INST}'
''').show()

┌─────────────────┬───────────────────┬──────────────┐
│ no_payment_date │ no_payment_amount │ both_missing │
│     int128      │      int128       │    int128    │
├─────────────────┼───────────────────┼──────────────┤
│            2905 │              2905 │         2905 │
└─────────────────┴───────────────────┴──────────────┘



Missed, or simply not paid yet when the data was collected? The due dates tell:

In [9]:
duckdb.sql(f'''
    SELECT
        MIN(DAYS_INSTALMENT) AS earliest_due,
        MAX(DAYS_INSTALMENT) AS latest_due,
        ROUND(AVG(DAYS_INSTALMENT), 0) AS avg_due
    FROM '{INST}'
    WHERE AMT_PAYMENT IS NULL
''').show()

┌──────────────┬────────────┬─────────┐
│ earliest_due │ latest_due │ avg_due │
│    double    │   double   │ double  │
├──────────────┼────────────┼─────────┤
│      -2908.0 │       -2.0 │  -779.0 │
└──────────────┴────────────┴─────────┘



**Findings**

- Only `AMT_PAYMENT` and `DAYS_ENTRY_PAYMENT` have blanks: **2,905 rows (0.02%)**, always on the same rows.
- Their due dates range from -2,908 to -2 days (average -779), spread across the whole history rather than bunched near the application date.
- These are **missed payments**: due, never paid.
- **A small percentage is not unimportant.** These rows are among the strongest risk signals in the table. Deleting them would hide non-payment; filling the date would make it look as if they paid.

---
## Q5. Ranges and anomalies

In [10]:
inst_summary[['column_name', 'min', 'max', 'avg']]

,column_name,min,max,avg
0,SK_ID_PREV,1000001,2843499,1903364.9695492988
1,SK_ID_CURR,100001,456255,278444.88173821557
2,NUM_INSTALMENT_VERSION,0.0,178.0,0.8566373016127933
3,NUM_INSTALMENT_NUMBER,1,277,18.87089583026623
4,DAYS_INSTALMENT,-2922.0,-1.0,-1042.2699916011295
5,DAYS_ENTRY_PAYMENT,-4921.0,-1.0,-1051.1136840621016
6,AMT_INSTALMENT,0.0,3771487.845,17050.906988970124
7,AMT_PAYMENT,0.0,3771487.845,17238.223249586794


Payments made more than a year before they were due, and zero amounts:

In [11]:
duckdb.sql(f'''
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT < -365 THEN 1 ELSE 0 END) AS paid_over_year_early,
        SUM(CASE WHEN AMT_INSTALMENT = 0 THEN 1 ELSE 0 END)                         AS zero_instalments,
        SUM(CASE WHEN AMT_PAYMENT = 0 THEN 1 ELSE 0 END)                            AS zero_payments
    FROM '{INST}'
''').show()

┌────────────┬──────────────────────┬──────────────────┬───────────────┐
│ total_rows │ paid_over_year_early │ zero_instalments │ zero_payments │
│   int64    │        int128        │      int128      │    int128     │
├────────────┼──────────────────────┼──────────────────┼───────────────┤
│   13605401 │                  392 │              290 │          1440 │
└────────────┴──────────────────────┴──────────────────┴───────────────┘



**Findings**

- `DAYS_INSTALMENT` reaches back -2,922 days (8 years), the same window as `bureau`. No positive dates, no negative amounts.
- `DAYS_ENTRY_PAYMENT` reaches back -4,921 days (13.5 years): **392 payments (0.003%)** were made more than a year before they were due. Likely errors; capped later.
- **290 zero instalments (0.002%):** a scheduled payment of nothing. Ignored.
- **1,440 zero payments (0.01%):** "paid nothing". Together with the 2,905 NULLs, **"no payment" is recorded two ways**. SQL skips NULLs but counts zeros, so both must be handled together when building a missed-payment feature.

---
## Summary

| Question | `installments_payments` |
|---|---|
| Grain | One row per **payment** (13.6M), not per instalment: split payments and schedule versions |
| Real rows | Paid minus due = days late; one payment can be recorded on two version rows |
| Types | All numeric; due vs paid pairs; version is a label |
| Missing | 2,905 missed payments (both payment columns blank) |
| Anomalies | 392 payments over a year early; "no payment" stored as NULL and as 0 |

**Carried into feature engineering:** aggregate payments to instalment level first (without double-counting across versions), then to customer level; compute days late as paid minus due; treat NULL and 0 payments together as missed.